In [1]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import json
import math
import random
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from peft import PeftModel
    HAS_PEFT = True
except Exception:
    HAS_PEFT = False

# ============================================================
# CONFIG
# ============================================================

BASE_MODEL = "microsoft/phi-3-mini-4k-instruct"
SFT_PATH = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"

OUT_DIR = "/kaggle/working/phi3_two_pass_self_correction"
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
REPORTS_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

SEED = 42
EVAL_SEED = 42

# Reasoning tasks where self-correction is most informative.
RUN_GSM8K = True
RUN_STRATEGYQA = True
RUN_MMLU = False  # optional, off by default for this experiment

N_GSM8K = 60
N_STRATEGYQA = 60
N_MMLU_PER_SUBJECT = 15
MMLU_SUBJECTS = [
    "abstract_algebra",
    "college_mathematics",
    "logical_fallacies",
]

# Two-pass generation settings.
PASS1_MAX_NEW = {
    "gsm8k": 256,
    "strategyqa": 48,
    "mmlu": 48,
}
PASS2_MAX_NEW = {
    "gsm8k": 256,
    "strategyqa": 48,
    "mmlu": 48,
}
TEMPERATURE = 0.7
TOP_P = 0.9

# If True, use task-specific few-shot style in pass 1 and critique instructions in pass 2.
USE_FEWSHOT = True

# Use a critique marker between passes.
CRITIQUE_TOKEN = "<critique>"
ANSWER_TOKEN = "<answer>"
THINK_TOKEN = "<think>"

# If the first pass is already correct, we still run pass 2 to see if it regresses.
RUN_ALL_SAMPLES = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

TOKENIZER = None
MODEL = None

# ============================================================
# UTILITIES
# ============================================================

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def save_df(df, path):
    ensure_dir(os.path.dirname(path))
    df.to_csv(path, index=False)

def save_json(obj, path):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def preview_text(s, max_len=220):
    s = str(s).replace("\n", " ")
    return s[:max_len] + ("..." if len(s) > max_len else "")

def to_jsonable(x):
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return json.dumps(str(x), ensure_ascii=False)

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x).strip().lower())

def normalize_number(x):
    if x is None:
        return None
    try:
        return float(re.sub(r"[$,]", "", str(x)))
    except Exception:
        return None

def same_numeric(a, b, tol=1e-6):
    na = normalize_number(a)
    nb = normalize_number(b)
    if na is None or nb is None:
        return False
    return abs(na - nb) <= tol

def canonical_number_str(x):
    v = normalize_number(x)
    if v is None:
        return None
    if abs(v - round(v)) < 1e-6:
        return str(int(round(v)))
    return f"{v:.6f}".rstrip("0").rstrip(".")

# ============================================================
# ADVANCED PLOTTING UTILITIES
# ============================================================

def bar_plot(labels, values, title, path, ylabel="Value"):
    plt.figure(figsize=(9, 4))
    plt.bar(labels, values, color='#4C72B0')
    plt.title(title)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def line_plot(x, ys, labels, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(10, 5))
    for y, label in zip(ys, labels):
        plt.plot(x, y, marker="o", linewidth=1.6, label=label)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def histogram(values, title, path, xlabel="Value", bins=12):
    plt.figure(figsize=(8.8, 4.2))
    plt.hist(values, bins=bins, color='#DD8452', edgecolor='black', alpha=0.8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def heatmap(mat, xlabels, ylabels, title, path, xlabel="", ylabel="", cmap="viridis"):
    plt.figure(figsize=(max(8, len(xlabels) * 0.55), max(5, len(ylabels) * 0.38)))
    im = plt.imshow(mat, aspect="auto", cmap=cmap)
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    plt.yticks(range(len(ylabels)), ylabels)
    for i in range(len(ylabels)):
        for j in range(len(xlabels)):
            val = mat[i, j]
            color = "white" if abs(val) < (np.nanmax(np.abs(mat)) / 2) else "black"
            if not np.isnan(val):
                plt.text(j, i, f"{val:.2f}", ha="center", va="center", color=color)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def scatter(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(6.8, 5.2))
    plt.scatter(x, y, alpha=0.8, color='#55A868')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def hexbin_plot(x, y, title, path, xlabel="X", ylabel="Y"):
    plt.figure(figsize=(8, 6))
    hb = plt.hexbin(x, y, gridsize=15, cmap='inferno', mincnt=1)
    cb = plt.colorbar(hb)
    cb.set_label('Density')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def violin_split_plot(df, val_col, split_col, title, path, ylabel="Value"):
    categories = sorted(df[split_col].unique())
    data = [df[df[split_col] == cat][val_col].dropna().values for cat in categories]
    
    plt.figure(figsize=(8, 5))
    if any(len(d) > 0 for d in data):
        parts = plt.violinplot(data, positions=range(len(categories)), showmeans=True, showextrema=True)
        for pc in parts['bodies']:
            pc.set_facecolor('#4C72B0')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.title(title)
        plt.xticks(range(len(categories)), categories)
        plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def correlation_heatmap(df, cols, title, path):
    df_numeric = df.copy()
    for col in cols:
        if df_numeric[col].dtype == bool or df_numeric[col].dtype == object:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
    
    sub_df = df_numeric[cols].dropna()
    if len(sub_df) < 2: return
    
    corr = sub_df.corr().values
    plt.figure(figsize=(8, 6))
    im = plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    plt.colorbar(im)
    plt.xticks(range(len(cols)), cols, rotation=30, ha='right')
    plt.yticks(range(len(cols)), cols)
    for i in range(len(cols)):
        for j in range(len(cols)):
            color = 'black' if abs(corr[i, j]) < 0.5 else 'white'
            plt.text(j, i, f"{corr[i, j]:.2f}", ha='center', va='center', color=color)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

def radar_chart(categories, values_dict, title, path):
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
    
    plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    
    plt.xticks(angles[:-1], categories, size=10)
    ax.set_rlabel_position(0)
    
    colors = ['#4C72B0', '#C44E52', '#55A868']
    for idx, (label, values) in enumerate(values_dict.items()):
        vals = list(values)
        vals += vals[:1]
        ax.plot(angles, vals, linewidth=2, linestyle='solid', label=label, color=colors[idx % len(colors)])
        ax.fill(angles, vals, alpha=0.25, color=colors[idx % len(colors)])
        
    plt.title(title, size=15, y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# EXTRACTION UTILS
# ============================================================

def extract_number(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    span = re.sub(r"[$,]", "", span)
    nums = re.findall(r"-?\d+\.?\d*", span)
    if nums:
        return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", re.sub(r"[$,]", "", text))
    return nums[-1] if nums else None

def extract_yes_no(text):
    if text is None:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.IGNORECASE | re.DOTALL)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None

def extract_mcq(text, choices=None):
    if text is None:
        return None
    span = text
    m = re.findall(r"<answer>(.*?)</answer>", span, flags=re.IGNORECASE | re.DOTALL)
    if m:
        span = m[-1]
    span_up = span.upper().strip()
    if span_up in ("A", "B", "C", "D"):
        return span_up
    patterns = [
        r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?",
        r"\b([ABCD])\b\s*$",
        r"<ANSWER>\s*\(?([ABCD])\)?",
    ]
    for pat in patterns:
        hits = re.findall(pat, span_up)
        if hits:
            return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", span_up)
    if hits:
        return hits[-1].upper()
    if choices is not None:
        low = span.lower()
        for i, c in enumerate(choices):
            if str(c).strip().lower() in low:
                return chr(65 + i)
    return None

def exact_correct(task, pred, gold):
    if task == "gsm8k":
        return same_numeric(pred, gold)
    return pred == gold

def letter_to_content(letter, choices):
    if letter is None:
        return None
    letter = letter.upper().strip()
    if letter not in "ABCD":
        return None
    idx = ord(letter) - 65
    if idx < 0 or idx >= len(choices):
        return None
    return choices[idx]

# ============================================================
# CACHE
# ============================================================

def cached_indices(name, ds_len, n, seed=EVAL_SEED):
    ensure_dir(CACHE_DIR)
    n = min(n, ds_len)
    path = os.path.join(CACHE_DIR, f"{name}_n{n}_seed{seed}.json")
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(ds_len, size=n, replace=False).tolist()
    with open(path, "w") as f:
        json.dump(idx, f)
    return idx

def sample_dataset(ds, name, n, seed=EVAL_SEED):
    idx = cached_indices(name, len(ds), n, seed=seed)
    return ds.select(idx), idx

# ============================================================
# MODEL LOADING
# ============================================================

def load_tokenizer():
    global TOKENIZER
    if TOKENIZER is None:
        src = SFT_PATH if (HAS_PEFT and os.path.exists(SFT_PATH)) else BASE_MODEL
        TOKENIZER = AutoTokenizer.from_pretrained(src)
        if TOKENIZER.pad_token is None:
            TOKENIZER.pad_token = TOKENIZER.eos_token
    return TOKENIZER

def load_model():
    global MODEL
    kwargs = {"torch_dtype": DTYPE}
    try:
        kwargs["attn_implementation"] = "eager"
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)
    except Exception:
        kwargs.pop("attn_implementation", None)
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs)

    model = model.to(DEVICE)
    model.eval()

    if HAS_PEFT and os.path.exists(SFT_PATH):
        try:
            model = PeftModel.from_pretrained(model, SFT_PATH).merge_and_unload().to(DEVICE)
            model.eval()
        except Exception:
            pass

    try:
        model.config.use_cache = False
    except Exception:
        pass

    MODEL = model
    print(f"Loaded model successfully.")
    return model, load_tokenizer()

# ============================================================
# DATA
# ============================================================

def load_gsm8k_samples(n=N_GSM8K):
    ds = load_dataset("gsm8k", "main", split="test")
    ds, idx = sample_dataset(ds, "gsm8k_self_correction", n)
    samples = []
    for i, s in enumerate(ds):
        ans = s["answer"]
        rationale, gold = ans.split("####", 1)
        gold = canonical_number_str(gold.strip())
        samples.append({
            "sample_id": i,
            "source_index": idx[i],
            "task": "gsm8k",
            "question": s["question"],
            "gold": gold,
            "gold_rationale": rationale.strip(),
        })
    return samples

def load_strategyqa_samples(n=N_STRATEGYQA):
    ds = load_dataset("ChilleD/StrategyQA", split="test")
    ds, idx = sample_dataset(ds, "strategyqa_self_correction", n)
    samples = []
    for i, s in enumerate(ds):
        gold = "yes" if bool(s["answer"]) else "no"
        samples.append({
            "sample_id": i,
            "source_index": idx[i],
            "task": "strategyqa",
            "question": s["question"],
            "gold": gold,
        })
    return samples

def load_mmlu_samples(subjects=MMLU_SUBJECTS, n_per_subject=N_MMLU_PER_SUBJECT):
    all_samples = []
    for subject in subjects:
        ds = load_dataset("cais/mmlu", subject, split="test")
        ds, idx = sample_dataset(ds, f"mmlu_self_correction_{subject}", n_per_subject)
        for i, s in enumerate(ds):
            all_samples.append({
                "sample_id": len(all_samples),
                "source_index": idx[i],
                "task": "mmlu",
                "subject": subject,
                "question": s["question"],
                "choices": list(s["choices"]),
                "gold": chr(65 + int(s["answer"])),
            })
    return all_samples

# ============================================================
# PROMPTS
# ============================================================

def build_pass1_prompt(sample):
    task = sample["task"]
    if task == "gsm8k":
        return (
            f"Question: {sample['question']}\n"
            f"{THINK_TOKEN} Solve carefully step by step {THINK_TOKEN}\n"
            f"{ANSWER_TOKEN}"
        )
    if task == "strategyqa":
        return (
            f"Question: {sample['question']}\n"
            f"{THINK_TOKEN} Reason carefully and decide yes or no {THINK_TOKEN}\n"
            f"{ANSWER_TOKEN}"
        )
    if task == "mmlu":
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
        return (
            f"## Question: {sample['question']}\n"
            f"{opts}\n"
            f"{THINK_TOKEN} Choose the best option {THINK_TOKEN}\n"
            f"{ANSWER_TOKEN}"
        )
    raise ValueError(task)

def build_pass2_prompt(sample, pass1_completion, pass1_prediction):
    task = sample["task"]
    if task == "gsm8k":
        return (
            f"Question: {sample['question']}\n\n"
            f"Previous attempt:\n{pass1_completion}\n\n"
            f"{CRITIQUE_TOKEN} Review the previous attempt and point out any arithmetic or reasoning mistakes. {CRITIQUE_TOKEN}\n"
            f"Then provide a corrected solution and finish with ONLY the final numeric answer inside {ANSWER_TOKEN}."
        )
    if task == "strategyqa":
        return (
            f"Question: {sample['question']}\n\n"
            f"Previous answer: {pass1_prediction}\n"
            f"Previous attempt text:\n{pass1_completion}\n\n"
            f"{CRITIQUE_TOKEN} Review the previous attempt and explain whether it should be yes or no. {CRITIQUE_TOKEN}\n"
            f"Then provide the corrected answer and finish with ONLY yes or no inside {ANSWER_TOKEN}."
        )
    if task == "mmlu":
        opts = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample["choices"])])
        return (
            f"## Question: {sample['question']}\n"
            f"{opts}\n\n"
            f"Previous answer: {pass1_prediction}\n"
            f"Previous attempt text:\n{pass1_completion}\n\n"
            f"{CRITIQUE_TOKEN} Reconsider the options carefully and identify any mismatch in the previous answer. {CRITIQUE_TOKEN}\n"
            f"Then provide the corrected answer and finish with ONLY one letter (A, B, C, or D) inside {ANSWER_TOKEN}."
        )
    raise ValueError(task)

# ============================================================
# GENERATION
# ============================================================

@torch.inference_mode()
def generate(prompt, max_new_tokens, temperature=TEMPERATURE, top_p=TOP_P):
    inputs = TOKENIZER(prompt, return_tensors="pt").to(DEVICE)
    out = MODEL.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0 else False,
        temperature=temperature if temperature > 0 else None,
        top_p=top_p if temperature > 0 else None,
        pad_token_id=TOKENIZER.eos_token_id,
        eos_token_id=TOKENIZER.eos_token_id,
    )
    full_output = TOKENIZER.decode(out[0], skip_special_tokens=True)
    prompt_len = inputs["input_ids"].shape[1]
    completion = TOKENIZER.decode(out[0][prompt_len:], skip_special_tokens=True)
    return full_output, completion

def parse_by_task(task, text, choices=None):
    if task == "gsm8k":
        return canonical_number_str(extract_number(text))
    if task == "strategyqa":
        return extract_yes_no(text)
    if task == "mmlu":
        return extract_mcq(text, choices=choices)
    raise ValueError(task)

def task_correct(task, pred, gold):
    if task == "gsm8k":
        return same_numeric(pred, gold)
    return pred == gold

def commitment_score(text, task):
    t = normalize_text(text)
    score = 0.5
    if "<answer>" in t and "</answer>" in t:
        score += 0.15
    if "<think>" in t:
        score += 0.1
    if task == "strategyqa":
        if re.search(r"\b(yes|no)\b", t):
            score += 0.15
        if any(h in t for h in ["maybe", "probably", "perhaps", "not sure", "uncertain", "depends"]):
            score -= 0.2
    elif task == "gsm8k":
        if re.search(r"\d", t):
            score += 0.1
    elif task == "mmlu":
        if re.search(r"\b([ABCD])\b", t):
            score += 0.1
    if len(t.split()) <= 6:
        score += 0.1
    if len(t.split()) > 30:
        score -= 0.1
    return max(0.0, min(1.0, score))

# ============================================================
# TWO-PASS PIPELINE
# ============================================================

def run_two_pass(sample):
    task = sample["task"]
    choices = sample.get("choices")
    gold = sample["gold"]

    p1_prompt = build_pass1_prompt(sample)
    p1_full, p1_completion = generate(p1_prompt, PASS1_MAX_NEW[task])
    p1_pred = parse_by_task(task, p1_completion, choices=choices)
    p1_correct = task_correct(task, p1_pred, gold)
    p1_commit = commitment_score(p1_completion, task)
    p1_tokens = len(TOKENIZER(p1_completion).input_ids)

    # Always run pass2 so we can measure regressions too.
    p2_prompt = build_pass2_prompt(sample, p1_completion, p1_pred)
    p2_full, p2_completion = generate(p2_prompt, PASS2_MAX_NEW[task])
    p2_pred = parse_by_task(task, p2_completion, choices=choices)
    p2_correct = task_correct(task, p2_pred, gold)
    p2_commit = commitment_score(p2_completion, task)
    p2_tokens = len(TOKENIZER(p2_completion).input_ids)

    improvement = int(p2_correct and not p1_correct)
    regression = int(p1_correct and not p2_correct)
    unchanged = int(p1_correct == p2_correct)

    first_status = "correct" if p1_correct else "wrong"
    second_status = "correct" if p2_correct else "wrong"
    transition = f"{first_status}->{second_status}"

    # For human-readable text analysis.
    p1_hedged = bool(re.search(r"\b(maybe|probably|perhaps|not sure|uncertain|depends)\b", normalize_text(p1_completion)))
    p2_hedged = bool(re.search(r"\b(maybe|probably|perhaps|not sure|uncertain|depends)\b", normalize_text(p2_completion)))

    row = {
        "task": task,
        "subject": sample.get("subject", ""),
        "sample_id": sample["sample_id"],
        "source_index": sample["source_index"],
        "question": sample["question"],
        "choices": to_jsonable(choices),
        "gold": gold,
        "pass1_prompt": p1_prompt,
        "pass1_full_output": p1_full,
        "pass1_completion": p1_completion,
        "pass1_prediction": p1_pred,
        "pass1_correct": bool(p1_correct),
        "pass1_commitment": p1_commit,
        "pass1_tokens": p1_tokens,
        "pass1_hedged": bool(p1_hedged),
        "pass2_prompt": p2_prompt,
        "pass2_full_output": p2_full,
        "pass2_completion": p2_completion,
        "pass2_prediction": p2_pred,
        "pass2_correct": bool(p2_correct),
        "pass2_commitment": p2_commit,
        "pass2_tokens": p2_tokens,
        "pass2_hedged": bool(p2_hedged),
        "improved": bool(improvement),
        "regressed": bool(regression),
        "unchanged": bool(unchanged),
        "transition": transition,
        "pass1_vs_pass2_prediction_changed": bool(p1_pred != p2_pred),
        "pass1_vs_pass2_text_changed": bool(normalize_text(p1_completion) != normalize_text(p2_completion)),
        "pass1_text_len": len(p1_completion),
        "pass2_text_len": len(p2_completion),
    }
    return row

# ============================================================
# ANALYSIS / PLOTS
# ============================================================

def summarize_task(df):
    if len(df) == 0:
        return {}
    return {
        "n": int(len(df)),
        "pass1_accuracy": float(df["pass1_correct"].mean()),
        "pass2_accuracy": float(df["pass2_correct"].mean()),
        "improvement_rate": float(df["improved"].mean()),
        "regression_rate": float(df["regressed"].mean()),
        "unchanged_rate": float(df["unchanged"].mean()),
        "prediction_change_rate": float((df["pass1_prediction"] != df["pass2_prediction"]).mean()),
        "text_change_rate": float(df["pass1_vs_pass2_text_changed"].mean()),
        "pass1_mean_commitment": float(df["pass1_commitment"].mean()),
        "pass2_mean_commitment": float(df["pass2_commitment"].mean()),
        "pass1_mean_tokens": float(df["pass1_tokens"].mean()),
        "pass2_mean_tokens": float(df["pass2_tokens"].mean()),
        "pass1_mean_text_len": float(df["pass1_text_len"].mean()),
        "pass2_mean_text_len": float(df["pass2_text_len"].mean()),
    }

def transition_matrix(df, task):
    labels = ["wrong", "correct"]
    mat = pd.DataFrame(0, index=labels, columns=labels)
    for _, r in df.iterrows():
        a = "correct" if bool(r["pass1_correct"]) else "wrong"
        b = "correct" if bool(r["pass2_correct"]) else "wrong"
        mat.loc[a, b] += 1
    return mat

def plot_reliability(df, title, path, n_bins=10):
    if len(df) == 0:
        return
    bins = np.linspace(0, 1, n_bins + 1)
    ids = np.digitize(df["pass2_commitment"].to_numpy(dtype=np.float64), bins) - 1
    rows = []
    for b in range(n_bins):
        mask = ids == b
        if mask.sum() == 0:
            rows.append({"bin": b, "conf": np.nan, "acc": np.nan})
            continue
        rows.append({
            "bin": b,
            "conf": float(df.loc[mask, "pass2_commitment"].mean()),
            "acc": float(df.loc[mask, "pass2_correct"].mean()),
        })
    r = pd.DataFrame(rows)
    r = r.dropna()
    if len(r) == 0:
        return
    plt.figure(figsize=(6.5, 5.5))
    plt.plot([0, 1], [0, 1], linestyle="--", label="ideal")
    plt.plot(r["conf"], r["acc"], marker="o", label="model")
    plt.xlabel("Pass-2 commitment score")
    plt.ylabel("Pass-2 accuracy")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

# ============================================================
# MAIN
# ============================================================

def main():
    print("Loading model and tokenizer ...")
    load_model()

    tasks = []
    if RUN_GSM8K:
        tasks.append("gsm8k")
    if RUN_STRATEGYQA:
        tasks.append("strategyqa")
    if RUN_MMLU:
        tasks.append("mmlu")

    all_rows = []
    summaries = []

    for task in tasks:
        if task == "gsm8k":
            samples = load_gsm8k_samples(N_GSM8K)
        elif task == "strategyqa":
            samples = load_strategyqa_samples(N_STRATEGYQA)
        elif task == "mmlu":
            samples = load_mmlu_samples(MMLU_SUBJECTS, N_MMLU_PER_SUBJECT)
        else:
            raise ValueError(task)

        print(f"\n=== Running {task.upper()} on {len(samples)} samples ===")
        task_rows = []
        for i, sample in enumerate(samples):
            print(f"  {i+1}/{len(samples)} | {preview_text(sample['question'], 90)}")
            row = run_two_pass(sample)
            task_rows.append(row)
            all_rows.append(row)
            free_memory()

        df = pd.DataFrame(task_rows)
        save_df(df, os.path.join(CSV_DIR, f"{task}_two_pass_results.csv"))

        summ = summarize_task(df)
        summ["task"] = task
        summaries.append(summ)
        save_json(summ, os.path.join(REPORTS_DIR, f"{task}_summary.json"))

        if not df.empty:
            bar_plot(
                ["pass1", "pass2"],
                [summ["pass1_accuracy"], summ["pass2_accuracy"]],
                f"{task.upper()} accuracy: pass 1 vs pass 2",
                os.path.join(PLOTS_DIR, f"{task}_accuracy_bar.png"),
                ylabel="Accuracy",
            )
            bar_plot(
                ["improved", "regressed", "unchanged"],
                [summ["improvement_rate"], summ["regression_rate"], summ["unchanged_rate"]],
                f"{task.upper()} outcome transitions",
                os.path.join(PLOTS_DIR, f"{task}_transition_rates.png"),
                ylabel="Rate",
            )
            bar_plot(
                ["pass1", "pass2"],
                [summ["pass1_mean_commitment"], summ["pass2_mean_commitment"]],
                f"{task.upper()} commitment: pass 1 vs pass 2",
                os.path.join(PLOTS_DIR, f"{task}_commitment_bar.png"),
                ylabel="Commitment score",
            )
            line_plot(
                [0, 1],
                [[summ["pass1_accuracy"], summ["pass2_accuracy"]]],
                ["accuracy"],
                f"{task.upper()} pass-to-pass improvement",
                os.path.join(PLOTS_DIR, f"{task}_accuracy_line.png"),
                xlabel="Pass",
                ylabel="Accuracy",
            )

            histogram(
                df["pass1_text_len"].tolist(),
                f"{task.upper()} pass-1 completion length",
                os.path.join(PLOTS_DIR, f"{task}_pass1_length_hist.png"),
                xlabel="Characters",
                bins=12,
            )
            histogram(
                df["pass2_text_len"].tolist(),
                f"{task.upper()} pass-2 completion length",
                os.path.join(PLOTS_DIR, f"{task}_pass2_length_hist.png"),
                xlabel="Characters",
                bins=12,
            )
            histogram(
                (df["pass2_commitment"] - df["pass1_commitment"]).tolist(),
                f"{task.upper()} commitment delta (pass2 - pass1)",
                os.path.join(PLOTS_DIR, f"{task}_commitment_delta_hist.png"),
                xlabel="Commitment delta",
                bins=12,
            )
            
            # Advanced Plot 1: Hexbin maps instead of overlapping scatter plots
            hexbin_plot(
                df["pass1_commitment"].tolist(),
                df["pass2_commitment"].tolist(),
                f"{task.upper()} Commitment Shift Density Map",
                os.path.join(PLOTS_DIR, f"{task}_commitment_hexbin.png"),
                xlabel="Pass-1 commitment",
                ylabel="Pass-2 commitment",
            )

            # Advanced Plot 2: Violin plot conditioning
            violin_split_plot(
                df,
                "pass2_commitment",
                "transition",
                f"{task.upper()} Pass 2 Commitment split by Transition Outcome",
                os.path.join(PLOTS_DIR, f"{task}_commitment_transition_violin.png"),
                ylabel="Pass-2 Commitment"
            )

            # Advanced Plot 3: Cross Metric Correlation Matrix
            correlation_heatmap(
                df,
                ["pass1_tokens", "pass2_tokens", "pass1_commitment", "pass2_commitment", "improved", "regressed"],
                f"{task.upper()} Two-Pass Metric Correlations",
                os.path.join(PLOTS_DIR, f"{task}_metric_correlations.png")
            )

            trans = transition_matrix(df, task)
            save_df(trans.reset_index().rename(columns={"index": "pass1"}), os.path.join(CSV_DIR, f"{task}_transition_matrix.csv"))
            heatmap(
                trans.values,
                trans.columns.tolist(),
                trans.index.tolist(),
                f"{task.upper()} correctness transition matrix",
                os.path.join(PLOTS_DIR, f"{task}_transition_heatmap.png"),
                xlabel="Pass-2",
                ylabel="Pass-1",
                cmap="Blues",
            )

            plot_reliability(
                df,
                f"{task.upper()} reliability (pass 2 commitment)",
                os.path.join(PLOTS_DIR, f"{task}_reliability.png"),
            )

            trans_rows = []
            for _, r in df.iterrows():
                trans_rows.append({
                    "task": task,
                    "subject": r.get("subject", ""),
                    "sample_id": r["sample_id"],
                    "question": r["question"],
                    "gold": r["gold"],
                    "transition": r["transition"],
                    "pass1_prediction": r["pass1_prediction"],
                    "pass2_prediction": r["pass2_prediction"],
                    "pass1_correct": r["pass1_correct"],
                    "pass2_correct": r["pass2_correct"],
                    "improved": r["improved"],
                    "regressed": r["regressed"],
                    "pass1_commitment": r["pass1_commitment"],
                    "pass2_commitment": r["pass2_commitment"],
                    "pass1_tokens": r["pass1_tokens"],
                    "pass2_tokens": r["pass2_tokens"],
                    "pass1_hedged": r["pass1_hedged"],
                    "pass2_hedged": r["pass2_hedged"],
                    "pass1_full_output": r["pass1_full_output"],
                    "pass2_full_output": r["pass2_full_output"],
                    "pass1_completion": r["pass1_completion"],
                    "pass2_completion": r["pass2_completion"],
                })
            save_df(pd.DataFrame(trans_rows), os.path.join(CSV_DIR, f"{task}_transition_details.csv"))

    all_df = pd.DataFrame(all_rows)
    summary_df = pd.DataFrame(summaries)

    save_df(all_df, os.path.join(CSV_DIR, "all_two_pass_results.csv"))
    save_df(summary_df, os.path.join(CSV_DIR, "summary.csv"))
    save_json(summaries, os.path.join(REPORTS_DIR, "summary.json"))

    if not summary_df.empty:
        bar_plot(summary_df["task"].tolist(), summary_df["pass1_accuracy"].tolist(), "Pass-1 accuracy by task", os.path.join(PLOTS_DIR, "overall_pass1_accuracy.png"), ylabel="Accuracy")
        bar_plot(summary_df["task"].tolist(), summary_df["pass2_accuracy"].tolist(), "Pass-2 accuracy by task", os.path.join(PLOTS_DIR, "overall_pass2_accuracy.png"), ylabel="Accuracy")
        bar_plot(summary_df["task"].tolist(), summary_df["improvement_rate"].tolist(), "Improvement rate by task", os.path.join(PLOTS_DIR, "overall_improvement_rate.png"), ylabel="Rate")
        bar_plot(summary_df["task"].tolist(), summary_df["regression_rate"].tolist(), "Regression rate by task", os.path.join(PLOTS_DIR, "overall_regression_rate.png"), ylabel="Rate")
        bar_plot(summary_df["task"].tolist(), summary_df["pass1_mean_commitment"].tolist(), "Pass-1 commitment by task", os.path.join(PLOTS_DIR, "overall_pass1_commitment.png"), ylabel="Commitment")
        bar_plot(summary_df["task"].tolist(), summary_df["pass2_mean_commitment"].tolist(), "Pass-2 commitment by task", os.path.join(PLOTS_DIR, "overall_pass2_commitment.png"), ylabel="Commitment")

        # Advanced Plot 4: Radar chart comparing Pass 1 and Pass 2 per task
        radar_categories = ["Accuracy P1", "Accuracy P2", "Commitment P1", "Commitment P2", "Norm Tokens P1", "Norm Tokens P2"]
        radar_data = {}
        for t in summaries:
            max_tok = max(t["pass1_mean_tokens"], t["pass2_mean_tokens"])
            radar_data[t["task"]] = [
                t["pass1_accuracy"],
                t["pass2_accuracy"],
                t["pass1_mean_commitment"],
                t["pass2_mean_commitment"],
                t["pass1_mean_tokens"] / (max_tok + 1e-8),
                t["pass2_mean_tokens"] / (max_tok + 1e-8)
            ]
        if radar_data:
            radar_chart(
                radar_categories,
                radar_data,
                "Multi-Dimensional Two-Pass Critique Profiling",
                os.path.join(PLOTS_DIR, "overall_task_radar.png")
            )

    global_summary = {
        "n_rows": int(len(all_df)),
        "tasks": summary_df.to_dict(orient="records"),
        "global": {
            "pass1_accuracy": float(all_df["pass1_correct"].mean()) if len(all_df) else float("nan"),
            "pass2_accuracy": float(all_df["pass2_correct"].mean()) if len(all_df) else float("nan"),
            "improvement_rate": float(all_df["improved"].mean()) if len(all_df) else float("nan"),
            "regression_rate": float(all_df["regressed"].mean()) if len(all_df) else float("nan"),
            "unchanged_rate": float(all_df["unchanged"].mean()) if len(all_df) else float("nan"),
            "prediction_change_rate": float((all_df["pass1_prediction"] != all_df["pass2_prediction"]).mean()) if len(all_df) else float("nan"),
            "text_change_rate": float(all_df["pass1_vs_pass2_text_changed"].mean()) if len(all_df) else float("nan"),
            "mean_pass1_commitment": float(all_df["pass1_commitment"].mean()) if len(all_df) else float("nan"),
            "mean_pass2_commitment": float(all_df["pass2_commitment"].mean()) if len(all_df) else float("nan"),
        },
        "config": {
            "BASE_MODEL": BASE_MODEL,
            "SFT_PATH": SFT_PATH,
            "RUN_GSM8K": RUN_GSM8K,
            "RUN_STRATEGYQA": RUN_STRATEGYQA,
            "RUN_MMLU": RUN_MMLU,
            "N_GSM8K": N_GSM8K,
            "N_STRATEGYQA": N_STRATEGYQA,
            "N_MMLU_PER_SUBJECT": N_MMLU_PER_SUBJECT,
            "MMLU_SUBJECTS": MMLU_SUBJECTS,
            "PASS1_MAX_NEW": PASS1_MAX_NEW,
            "PASS2_MAX_NEW": PASS2_MAX_NEW,
            "TEMPERATURE": TEMPERATURE,
            "TOP_P": TOP_P,
            "USE_FEWSHOT": USE_FEWSHOT,
        },
    }
    save_json(global_summary, os.path.join(REPORTS_DIR, "global_summary.json"))

    md = ["# Two-pass self-correction experiment report\n"]
    md.append(f"- Model: `{BASE_MODEL}`")
    md.append(f"- Tasks: {', '.join(tasks)}")
    md.append(f"- Pass-1 max new tokens: {PASS1_MAX_NEW}")
    md.append(f"- Pass-2 max new tokens: {PASS2_MAX_NEW}")
    md.append(f"- Temperature: {TEMPERATURE}")
    md.append(f"- Top-p: {TOP_P}")
    md.append(f"- Few-shot enabled: {USE_FEWSHOT}")
    md.append("\n## Summary table\n")
    md.append("| Task | Pass 1 acc | Pass 2 acc | Improve | Regress | Unchanged |")
    md.append("|---|---:|---:|---:|---:|---:|")
    for _, r in summary_df.iterrows():
        md.append(
            f"| {r['task']} | {r['pass1_accuracy']:.3f} | {r['pass2_accuracy']:.3f} | {r['improvement_rate']:.3f} | {r['regression_rate']:.3f} | {r['unchanged_rate']:.3f} |"
        )
    md.append("\n## Interpretation hints\n")
    md.append("- Improvement cases justify the self-correction / process-reward direction.")
    md.append("- Regression cases show where the critique pass destabilizes already-correct answers.")
    md.append("- If the commit score rises in improved samples, the model is learning to be more decisive after review.")
    md.append("- Transition matrices show whether pass-2 mostly fixes wrong answers or also flips correct ones.")
    md.append("- The calibration-style reliability plots indicate whether the pass-2 commitment score tracks actual correctness.")

    with open(os.path.join(REPORTS_DIR, "report.md"), "w") as f:
        f.write("\n".join(md))

    print("\n===== SUMMARY =====")
    if not summary_df.empty:
        print(summary_df.to_string(index=False))
    print("\nGlobal metrics:")
    print(json.dumps(global_summary["global"], indent=2))
    print("\nSaved to:")
    print(f"- {OUT_DIR}")
    print(f"- CSVs: {CSV_DIR}")
    print(f"- Plots: {PLOTS_DIR}")
    print(f"- Reports: {REPORTS_DIR}")

    free_memory(MODEL)

if __name__ == "__main__":
    main()

Loading model and tokenizer ...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

data/train-00000-of-00001-506370352f6228(…):   0%|          | 0.00/369k [00:00<?, ?B/s]

data/test-00000-of-00001-bae602f3ee37f4c(…):   0%|          | 0.00/161k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1603 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/687 [00:00<?, ? examples/s]


=== Running STRATEGYQA on 60 samples ===
  1/60 | Could boolean algebra be described as binary?
  2/60 | Would lumberjacks get full after eating three dosa?
  3/60 | Would a member of the United States Air Force get a discount at Dunkin Donuts?
  4/60 | In a hypothetical race between a Swallow and an American Woodcock, would the Swallow win?
  5/60 | Were number of states in Ancient Greece underwhelming compared to US states in 1900?
  6/60 | Can petroleum jelly be used as fuel in a car?
  7/60 | Did Julia Roberts practice blast beats as a child?
  8/60 | Is myocardial infarction a brain problem?
  9/60 | Would a hedgehog avoid animals without a spinal cord?
  10/60 | Can The Hobbit be read in its entirety in four minutes?
  11/60 | Could Plato have agreed with the beliefs of Jainism?
  12/60 | Can Michael Bloomberg fund the debt of Micronesia for a decade?
  13/60 | Is Benjamin Franklin a prime candidate to have his statues removed by Black Lives Matter m...
  14/60 | Are sables rela